In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/howonmjeong/soldier-sleep-fatigue/sample_submission.csv
/kaggle/input/datasets/howonmjeong/soldier-sleep-fatigue/sleep_fatigue_test.csv
/kaggle/input/datasets/howonmjeong/soldier-sleep-fatigue/sleep_fatigue_answer.csv
/kaggle/input/datasets/howonmjeong/soldier-sleep-fatigue/sleep_fatigue.csv


In [2]:
path = '/kaggle/input/datasets/howonmjeong/soldier-sleep-fatigue/'
train = pd.read_csv(path+'sleep_fatigue.csv')
test = pd.read_csv(path+'sleep_fatigue_test.csv')
sub = pd.read_csv(path+'sample_submission.csv')

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

feats = ['avg_sleep_hours', 'night_shift_days', 'caffeine_intake']   # soldier_id 제외
nas = ['avg_sleep_hours', 'caffeine_intake']

print(train.isnull().sum())
print(train[feats].describe())

train[nas] = train[nas].fillna(train[nas].median())
ok = (train['avg_sleep_hours'].isna() | train['avg_sleep_hours'].between(1, 14)) & \
     (train['caffeine_intake'].isna() | train['caffeine_intake'].between(0, 1000))
train = train[ok].copy()

print(train.isnull().sum())
print(train[feats].describe())

soldier_id           0
avg_sleep_hours     12
night_shift_days     0
caffeine_intake     26
fatigue_score        0
dtype: int64
       avg_sleep_hours  night_shift_days  caffeine_intake
count       468.000000        480.000000       454.000000
mean          6.234615          2.456250       194.017621
std           1.471892          1.744111       164.733163
min           0.000000          0.000000         0.000000
25%           5.500000          1.000000       136.500000
50%           6.200000          2.000000       184.000000
75%           6.900000          4.000000       228.750000
max          23.000000          5.000000      2200.000000
soldier_id          0
avg_sleep_hours     0
night_shift_days    0
caffeine_intake     0
fatigue_score       0
dtype: int64
       avg_sleep_hours  night_shift_days  caffeine_intake
count       474.000000        474.000000       474.000000
mean          6.182489          2.455696       182.126582
std           1.041917          1.744559        68.65

In [4]:
X = train[feats]
y = train['fatigue_score']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

In [5]:
print(rmse)

4.2819364196812835
